In [1]:
import time
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from skimage.segmentation import slic

In [2]:
# Visible/NIR raster band order: B, G, Red, NIR
vis_nir = r"C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif"
# Red-edge/SWIR raster band order: RE1, RE2, SWIR1, SWIR2
re_swir = r"C:\AFLICM_Test\Imagery\reswir\reswir_mosaic_TL_sentinel2.tif"
# Segment output
SEG_FN = r"C:\AFLICM_Test\Imagery\reswir\Segments_10m.tif"
R, N = 3, 4
SWIR1 = 3
FEATURE_NAMES = ("Red", "NIR", "SWIR1", "NDVI")

TILE = 2048               # pixels per side, ~4M px per tile
PX_PER_SEGMENT = 30       # target object size (~0.3 ha at 10 m)
COMPACTNESS = 0.1         # tune on a test window (see below)

def make_features(vis_nir_arr, re_swir_arr):
    """Return (h, w, 4) Red/NIR/SWIR1/NDVI features and a valid-pixel mask.

    Inputs are masked reads of bands [Red, NIR] and [SWIR1], respectively.
    """
    red, nir = np.ma.asarray(vis_nir_arr, dtype="float32").filled(np.nan)
    swir1 = np.ma.asarray(re_swir_arr, dtype="float32").filled(np.nan)
    spectral = np.stack((red, nir, swir1), axis=0)
    valid = np.all(np.isfinite(spectral), axis=0) & np.all(spectral > 0, axis=0)
    ndvi = np.full(red.shape, np.nan, dtype="float32")
    np.divide(nir - red, nir + red, out=ndvi, where=(nir + red) != 0)
    features = np.stack((red, nir, swir1, ndvi), axis=-1)
    valid &= np.isfinite(ndvi)
    return features, valid


In [ ]:
with rasterio.open(vis_nir) as vis_src, rasterio.open(re_swir) as swir_src:
    if vis_src.crs is None or swir_src.crs is None:
        raise ValueError("Both input rasters must define a CRS")
    if vis_src.count < R or vis_src.count < N or swir_src.count < SWIR1:
        raise ValueError("Input rasters do not contain the requested feature bands")

    profile = vis_src.profile.copy()
    rows, cols = vis_src.height, vis_src.width
    with WarpedVRT(
        swir_src,
        crs=vis_src.crs,
        transform=vis_src.transform,
        width=cols,
        height=rows,
        resampling=Resampling.bilinear,
    ) as swir_aligned:
        # Estimate shared stretch limits once, on a decimated sample.
        step = 10
        sample_rows = max(1, (rows + step - 1) // step)
        sample_cols = max(1, (cols + step - 1) // step)
        small_vis = vis_src.read(
            [R, N], out_shape=(2, sample_rows, sample_cols), masked=True,
            resampling=Resampling.average,
        )
        small_swir = swir_aligned.read(
            SWIR1, out_shape=(sample_rows, sample_cols), masked=True,
            resampling=Resampling.average,
        )
        sample_features, sample_valid = make_features(small_vis, small_swir)
        if not sample_valid.any():
            raise ValueError("No valid pixels found in the sampled input rasters")
        lo = np.zeros(len(FEATURE_NAMES), dtype="float32")
        hi = np.ones(len(FEATURE_NAMES), dtype="float32")
        for k in range(len(FEATURE_NAMES)):
            lo[k], hi[k] = np.percentile(
                sample_features[..., k][sample_valid], (2, 98)
            )
        del small_vis, small_swir, sample_features, sample_valid
        print("feature order:", FEATURE_NAMES)
        print("stretch lo:", lo, "hi:", hi)

        profile.update(
            count=1, dtype="int32", nodata=0, compress="deflate",
            tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER",
        )
        offset = 0
        t0 = time.time()
        with rasterio.open(SEG_FN, "w", **profile) as dst:
            for r0 in range(0, rows, TILE):
                for c0 in range(0, cols, TILE):
                    win = Window(c0, r0, min(TILE, cols - c0), min(TILE, rows - r0))
                    vis_arr = vis_src.read([R, N], window=win, masked=True)
                    swir_arr = swir_aligned.read(SWIR1, window=win, masked=True)
                    feats, valid = make_features(vis_arr, swir_arr)

                    n_valid = int(valid.sum())
                    if n_valid < 100:
                        dst.write(np.zeros(valid.shape, dtype="int32"), 1, window=win)
                        continue

                    feats = np.clip((feats - lo) / (hi - lo + 1e-9), 0, 1)
                    feats[~valid] = 0
                    seg = slic(
                        feats,
                        n_segments=max(n_valid // PX_PER_SEGMENT, 1),
                        compactness=COMPACTNESS,
                        channel_axis=-1,
                        mask=valid,
                        start_label=1,
                        enforce_connectivity=True,
                        slic_zero=True,
                    ).astype("int32")
                    seg[~valid] = 0
                    seg[seg > 0] += offset
                    offset = max(offset, int(seg.max()))
                    dst.write(seg, 1, window=win)
                    print(
                        f"tile r{r0} c{c0}: total segments so far "
                        f"{offset:,} ({time.time() - t0:.0f}s)"
                    )


feature order: ('Red', 'NIR', 'SWIR1', 'NDVI')
stretch lo: [ 0.00755871  0.00594842  0.00647639 -0.17251183] hi: [0.15066808 0.36463666 0.35120717 0.8708859 ]
tile r0 c14336: total segments so far 69 (25s)
tile r0 c16384: total segments so far 81,593 (1694s)
tile r0 c18432: total segments so far 164,194 (3384s)
tile r2048 c14336: total segments so far 168,372 (3449s)


In [1]:
import os
import time
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from skimage.segmentation import slic
from concurrent.futures import ProcessPoolExecutor, as_completed

# Visible/NIR raster band order: B, G, Red, NIR
vis_nir = r"C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif"
# Red-edge/SWIR raster band order: RE1, RE2, SWIR1, SWIR2
re_swir = r"C:\AFLICM_Test\Imagery\reswir\reswir_mosaic_TL_sentinel2.tif"
SEG_FN = r"C:\AFLICM_Test\Imagery\reswir\Segments_10m.tif"

R, N = 3, 4
SWIR1 = 3
FEATURE_NAMES = ("Red", "NIR", "SWIR1", "NDVI")

TILE = 2048
PX_PER_SEGMENT = 30
COMPACTNESS = 0.1
SLIC_ZERO = False
MAX_ITER = 10                                # try 5 to trade a little accuracy for speed
WORKERS = max(1, (os.cpu_count() or 2) // 2) # ~1.5-2 GB RAM per worker; adjust to your machine
STRIDE = 250_000                             # ID range per tile; must exceed max segments per tile
ONLY_TILES = None                            # e.g. [(0, 16384)] to time a single tile


def make_features(vis_nir_arr, re_swir_arr):
    red, nir = np.ma.asarray(vis_nir_arr, dtype="float32").filled(np.nan)
    swir1 = np.ma.asarray(re_swir_arr, dtype="float32").filled(np.nan)
    spectral = np.stack((red, nir, swir1), axis=0)
    valid = np.all(np.isfinite(spectral), axis=0) & np.all(spectral > 0, axis=0)
    ndvi = np.full(red.shape, np.nan, dtype="float32")
    np.divide(nir - red, nir + red, out=ndvi, where=(nir + red) != 0)
    features = np.stack((red, nir, swir1, ndvi), axis=-1)
    valid &= np.isfinite(ndvi)
    return features, valid

In [ ]:
def prepare():
    """Stretch limits from a decimated read, plus output profile and raster size."""
    with rasterio.open(vis_nir) as vis_src, rasterio.open(re_swir) as swir_src:
        profile = vis_src.profile.copy()
        rows, cols = vis_src.height, vis_src.width
        with WarpedVRT(swir_src, crs=vis_src.crs, transform=vis_src.transform,
                       width=cols, height=rows,
                       resampling=Resampling.bilinear) as swir_aligned:
            step = 10
            sr, sc = max(1, (rows + step - 1) // step), max(1, (cols + step - 1) // step)
            small_vis = vis_src.read([R, N], out_shape=(2, sr, sc), masked=True,
                                     resampling=Resampling.average)
            small_swir = swir_aligned.read(SWIR1, out_shape=(sr, sc), masked=True,
                                           resampling=Resampling.average)
            f, v = make_features(small_vis, small_swir)
            if not v.any():
                raise ValueError("No valid pixels found in the sampled input rasters")
            lo = np.zeros(4, dtype="float32")
            hi = np.ones(4, dtype="float32")
            for k in range(4):
                lo[k], hi[k] = np.percentile(f[..., k][v], (2, 98))
    profile.update(count=1, dtype="int32", nodata=0, compress="deflate",
                   tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
    return lo, hi, profile, rows, cols


def do_tile(job):
    idx, r0, c0, lo, hi = job
    with rasterio.open(vis_nir) as vis_src, rasterio.open(re_swir) as swir_src:
        with WarpedVRT(swir_src, crs=vis_src.crs, transform=vis_src.transform,
                       width=vis_src.width, height=vis_src.height,
                       resampling=Resampling.bilinear) as swir_aligned:
            w = min(TILE, vis_src.width - c0)
            h = min(TILE, vis_src.height - r0)
            win = Window(c0, r0, w, h)
            vis_arr = vis_src.read([R, N], window=win, masked=True)
            swir_arr = swir_aligned.read(SWIR1, window=win, masked=True)

    feats, valid = make_features(vis_arr, swir_arr)
    n_valid = int(valid.sum())
    if n_valid < 100:
        return idx, r0, c0, w, h, None, 0

    feats = np.clip((feats - lo) / (hi - lo + 1e-9), 0, 1)
    feats[~valid] = 0
    seg = slic(
        feats,
        n_segments=max(n_valid // PX_PER_SEGMENT, 1),
        compactness=COMPACTNESS,
        max_num_iter=MAX_ITER,
        channel_axis=-1,
        mask=valid,
        start_label=1,
        enforce_connectivity=True,
        slic_zero=SLIC_ZERO,
    ).astype("int32")
    n_found = int(seg.max())
    if n_found >= STRIDE:
        raise ValueError(f"Tile {idx} has {n_found} segments; increase STRIDE")
    seg[~valid] = 0
    seg[seg > 0] += idx * STRIDE
    return idx, r0, c0, w, h, seg, n_found

feature order: ('Red', 'NIR', 'SWIR1', 'NDVI')
stretch lo: [ 0.00755871  0.00594842  0.00647639 -0.17251183] hi: [0.15066808 0.36463666 0.35120717 0.8708859 ]
171 tiles, 10 workers


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [3]:
lo, hi, profile, rows, cols = prepare()
print("feature order:", FEATURE_NAMES)
print("stretch lo:", lo, "hi:", hi)
coords = [(r, c) for r in range(0, rows, TILE) for c in range(0, cols, TILE)]
jobs = [(i, r, c, lo, hi) for i, (r, c) in enumerate(coords)
            if ONLY_TILES is None or (r, c) in ONLY_TILES]
print(len(jobs), "tiles,", WORKERS, "workers")
t0 = time.time()
done = total = 0
with rasterio.open(SEG_FN, "w", **profile) as dst, \
         ProcessPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(do_tile, j) for j in jobs]
        for f in as_completed(futs):
            idx, r0, c0, w, h, seg, n_found = f.result()
            if seg is not None:
                dst.write(seg, 1, window=Window(c0, r0, w, h))
                total += n_found
            done += 1
            print(f"{done}/{len(jobs)} tile r{r0} c{c0}: {n_found:,} segments "
                  f"(total {total:,}, {time.time() - t0:.0f}s)")

feature order: ('Red', 'NIR', 'SWIR1', 'NDVI')
stretch lo: [ 0.00755871  0.00594842  0.00647639 -0.17251183] hi: [0.15066808 0.36463666 0.35120717 0.8708859 ]
171 tiles, 10 workers


Traceback (most recent call last):
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
Traceback (most recent call last):
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is close

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.